**HOMEWORK 2**

**Student Name: Sai Siva Shankara Vara Prasad Kopparthi**

**Student ID: 700765221**

## Q5 Evaluation Metrics

In [2]:
import numpy as np

# Confusion matrix from the question
# Rows = System predictions, Columns = Gold labels
conf_matrix = np.array([
    [5, 10, 5],   # Predicted Cat
    [15, 20, 10], # Predicted Dog
    [0, 15, 10]   # Predicted Rabbit
])

classes = ["Cat", "Dog", "Rabbit"]

def compute_metrics(conf_matrix, classes):
    num_classes = len(classes)
    
    # Per-class precision and recall
    precisions = []
    recalls = []
    
    for i in range(num_classes):
        TP = conf_matrix[i, i]
        FP = np.sum(conf_matrix[i, :]) - TP
        FN = np.sum(conf_matrix[:, i]) - TP
        
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        
        precisions.append(precision)
        recalls.append(recall)
        
        print(f"{classes[i]} → Precision: {precision:.3f}, Recall: {recall:.3f}")
    
    # Macro-averaged
    macro_precision = np.mean(precisions)
    macro_recall = np.mean(recalls)
    
    # Micro-averaged
    TP_total = np.trace(conf_matrix)
    FP_total = np.sum(conf_matrix) - TP_total
    FN_total = FP_total  # in multi-class, FP = FN when pooling
    
    micro_precision = TP_total / (TP_total + FP_total)
    micro_recall = TP_total / (TP_total + FN_total)
    
    print("\nMacro Precision: {:.3f}, Macro Recall: {:.3f}".format(macro_precision, macro_recall))
    print("Micro Precision: {:.3f}, Micro Recall: {:.3f}".format(micro_precision, micro_recall))

# Run the function
compute_metrics(conf_matrix, classes)


Cat → Precision: 0.250, Recall: 0.250
Dog → Precision: 0.444, Recall: 0.444
Rabbit → Precision: 0.400, Recall: 0.400

Macro Precision: 0.365, Macro Recall: 0.365
Micro Precision: 0.389, Micro Recall: 0.389


## Q8 Bigram Language Model Implementation

In [3]:
from collections import Counter, defaultdict
import math

# Training corpus
corpus_sentences = [
    "<s> I love NLP </s>",
    "<s> I love deep learning </s>",
    "<s> deep learning is fun </s>"
]

# Tokenize
corpus = [sent.split() for sent in corpus_sentences]

# Compute unigram and bigram counts
unigram_counts = Counter()
bigram_counts = Counter()

for sent in corpus:
    for i, w in enumerate(sent):
        unigram_counts[w] += 1
        if i > 0:
            bigram = (sent[i - 1], w)
            bigram_counts[bigram] += 1

# Compute bigram probabilities (MLE)
bigram_totals = defaultdict(int)
for (w1, w2), c in bigram_counts.items():
    bigram_totals[w1] += c

bigram_probs = {}
for (w1, w2), c in bigram_counts.items():
    bigram_probs[(w1, w2)] = c / bigram_totals[w1]

# Function: compute sentence probability
def sentence_probability(tokens):
    prob = 1.0
    log_prob = 0.0
    for i in range(1, len(tokens)):
        w1, w2 = tokens[i - 1], tokens[i]
        count = bigram_counts.get((w1, w2), 0)
        denom = bigram_totals.get(w1, 0)
        if denom == 0:
            p = 0.0
        else:
            p = count / denom
        prob *= p
        if p == 0:
            log_prob = float("-inf")
        elif log_prob != float("-inf"):
            log_prob += math.log(p)
    return prob, log_prob

# Test sentences
s1 = "<s> I love NLP </s>".split()
s2 = "<s> I love deep learning </s>".split()

p1, lp1 = sentence_probability(s1)
p2, lp2 = sentence_probability(s2)

# Print results
print("=== Unigram Counts ===")
print(dict(unigram_counts))
print("\n=== Bigram Counts ===")
for k in sorted(bigram_counts):
    print(f"{k}: {bigram_counts[k]}")

print("\n=== Bigram Probabilities (MLE) ===")
for (w1, w2), p in sorted(bigram_probs.items()):
    print(f"P({w2}|{w1}) = {p:.3f}")

print("\n=== Sentence Evaluation ===")
print(f"S1: {' '.join(s1)} → P = {p1:.6f}, logP = {lp1:.6f}")
print(f"S2: {' '.join(s2)} → P = {p2:.6f}, logP = {lp2:.6f}")

print("\n=== Model Preference ===")
if p1 > p2:
    print("The model prefers S1 (<s> I love NLP </s>)")
elif p2 > p1:
    print("The model prefers S2 (<s> I love deep learning </s>)")
else:
    print("The model assigns equal probability to both sentences.")

=== Unigram Counts ===
{'<s>': 3, 'I': 2, 'love': 2, 'NLP': 1, '</s>': 3, 'deep': 2, 'learning': 2, 'is': 1, 'fun': 1}

=== Bigram Counts ===
('<s>', 'I'): 2
('<s>', 'deep'): 1
('I', 'love'): 2
('NLP', '</s>'): 1
('deep', 'learning'): 2
('fun', '</s>'): 1
('is', 'fun'): 1
('learning', '</s>'): 1
('learning', 'is'): 1
('love', 'NLP'): 1
('love', 'deep'): 1

=== Bigram Probabilities (MLE) ===
P(I|<s>) = 0.667
P(deep|<s>) = 0.333
P(love|I) = 1.000
P(</s>|NLP) = 1.000
P(learning|deep) = 1.000
P(</s>|fun) = 1.000
P(fun|is) = 1.000
P(</s>|learning) = 0.500
P(is|learning) = 0.500
P(NLP|love) = 0.500
P(deep|love) = 0.500

=== Sentence Evaluation ===
S1: <s> I love NLP </s> → P = 0.333333, logP = -1.098612
S2: <s> I love deep learning </s> → P = 0.166667, logP = -1.791759

=== Model Preference ===
The model prefers S1 (<s> I love NLP </s>)
